# Technology Diffusion Exact IP Validation

Run the exact influence-propagation IP on the same PA graph batches used by the exact script, then verify that the seed set it returns actually induces full spread.

In [ ]:
from __future__ import annotations

import itertools
import sys
import time
from pathlib import Path

import gurobipy as gp
import networkx as nx
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from technology_diffusion import (
    build_exact_ip,
    connected_component_spread,
    create_pa_graph,
)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

In [ ]:
def theta_const(g: nx.Graph, value: int) -> dict[int, int]:
    return {node: value for node in g.nodes()}


def theta_from_degrees(g: nx.Graph, rule: str = 'simple') -> dict[int, int]:
    if rule == 'simple':
        return {node: max(1, min(2, g.degree(node))) for node in g.nodes()}
    raise ValueError(f'Unknown rule: {rule}')


DEFAULT_C_LIST = [1, 2, 5]
DEFAULT_N_LIST = [6, 8, 10, 12, 15]
DEFAULT_SEED_LIST = [1, 42, 53, 99, 101]
DEFAULT_INIT_NODES = 5
DEFAULT_INIT_MODE = 'complete'


def build_cases() -> list[dict]:
    cases = []
    for n_nodes, c, seed in itertools.product(DEFAULT_N_LIST, DEFAULT_C_LIST, DEFAULT_SEED_LIST):
        g, thetas = create_pa_graph(
            n_nodes=n_nodes,
            c=c,
            seed=seed,
            init_nodes=DEFAULT_INIT_NODES,
            init_mode=DEFAULT_INIT_MODE,
        )
        cases.append(
            {
                'name': f'pa_n{n_nodes}_c{c}_seed{seed}',
                'n': n_nodes,
                'm': g.number_of_edges(),
                'c': c,
                'seed': seed,
                'init_nodes': DEFAULT_INIT_NODES,
                'init_mode': DEFAULT_INIT_MODE,
                'graph': g,
                'thetas': thetas,
            }
        )
    return cases


def solve_exact_case(case: dict, max_time: float = 60.0) -> dict:
    g: nx.Graph = case['graph']
    thetas: dict[int, int] = case['thetas']
    n_nodes = g.number_of_nodes()

    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 0)
    env.setParam('LogToConsole', 0)
    env.start()

    build_start = time.perf_counter()
    model, x, y, nvar = build_exact_ip(
        g,
        thetas,
        k=n_nodes,
        use_simultaneous=False,
        time_horizon=n_nodes,
        max_time=max_time,
        env=env,
    )
    build_time = time.perf_counter() - build_start

    if model is None:
        return {
            'name': case['name'],
            'n': n_nodes,
            'm': g.number_of_edges(),
            'c': case['c'],
            'seed': case['seed'],
            'status': 'build_failed',
            'build_time_s': round(build_time, 4),
            'solve_time_s': None,
            'objective_seeds': None,
            'seed_nodes': None,
            'full_spread': False,
            'spread': None,
            'history': None,
        }

    model.setParam('OutputFlag', 0)
    model.setParam('LogToConsole', 0)
    model.setParam('TimeLimit', max_time - build_time)
    model.optimize()

    solve_time = float(model.Runtime)
    total_time = build_time + solve_time

    seed_nodes = [node for node in g.nodes() if x[node, 0].X > 0.5]
    seed_vector = np.zeros(n_nodes, dtype=float)
    for node in seed_nodes:
        seed_vector[node] = 1.0

    spread, spread_hist, final_x = connected_component_spread(g, seed_vector, thetas, max_t=1000)
    objective_seeds = int(round(model.objVal)) if model.SolCount > 0 else None
    full_spread = spread == n_nodes

    return {
        'name': case['name'],
        'n': n_nodes,
        'm': g.number_of_edges(),
        'c': case['c'],
        'seed': case['seed'],
        'status': 'solved' if model.SolCount > 0 else 'no_solution',
        'build_time_s': round(build_time, 4),
        'solve_time_s': round(solve_time, 4),
        'total_time_s': round(total_time, 4),
        'objective_seeds': objective_seeds,
        'seed_nodes': seed_nodes,
        'spread': spread,
        'full_spread': full_spread,
        'spread_history': spread_hist,
    }

In [ ]:
cases = build_cases()
results = []
for index, case in enumerate(cases, start=1):
    print(f"[{index}/{len(cases)}] running {case['name']} (n={case['n']}, c={case['c']}, seed={case['seed']})")
    result = solve_exact_case(case, max_time=1200.0)
    results.append(result)
    print(f"[{index}/{len(cases)}] done {case['name']} -> spread={result['spread']} full_spread={result['full_spread']} seeds={result['objective_seeds']}")

summary = pd.DataFrame(results)
summary[['name', 'n', 'm', 'status', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread', 'build_time_s', 'solve_time_s']]

In [ ]:
# Print all cases where full_spread is False
failed = summary[~summary['full_spread']]
print(f'Failing cases: {len(failed)}')
failed[['name','n','c','seed','status','objective_seeds','seed_nodes','spread','full_spread', 'build_time_s', 'solve_time_s']]

In [ ]:
validation_results_dir = ROOT / "results" / "technology diffusion"
validation_results_dir.mkdir(parents=True, exist_ok=True)
failed.to_csv(
    validation_results_dir / "technology_diffusion_exact_ip_validation_failures.csv",
    index=False,
)

In [ ]:
all_good = bool(summary['full_spread'].all())
full_spread_ok = bool(summary['full_spread'].all())

print(f'All cases full spread: {full_spread_ok}')
print(f'Overall sanity check passed: {all_good}')

summary[['name', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread']]

In [ ]:
def brute_force_spread(g: nx.Graph, thetas: dict[int, int]) -> int:
    for b in range(g.number_of_nodes()):
        for idx in itertools.combinations(g.nodes(), b):
            seed_vector = np.zeros(g.number_of_nodes(), dtype=float)
            for node in idx:
                seed_vector[node] = 1.0
            spread, _, _ = connected_component_spread(g, seed_vector, thetas, max_t=1000)
            if spread == g.number_of_nodes():
                return b
    return g.number_of_nodes()

In [ ]:
cases = build_cases()
for case in cases:
    b = brute_force_spread(case['graph'], case['thetas'])
    print(f"{case['name']}: brute-force spread = {b}")

In [ ]:
# Inspect one critical instance with solver output enabled.
# Edit the parameters below to target the graph you want to inspect.
critical_n = 12
critical_c = 1
critical_seed = 99
critical_init_nodes = DEFAULT_INIT_NODES
critical_init_mode = DEFAULT_INIT_MODE
critical_max_time = 600.0

critical_graph, critical_thetas = create_pa_graph(
    n_nodes=critical_n,
    c=critical_c,
    seed=critical_seed,
    init_nodes=critical_init_nodes,
    init_mode=critical_init_mode,
)

critical_env = gp.Env(empty=True)
critical_env.setParam('OutputFlag', 1)
critical_env.setParam('LogToConsole', 1)
critical_env.start()

critical_start = time.perf_counter()
critical_model, critical_x, critical_y, critical_nvar = build_exact_ip(
    critical_graph,
    critical_thetas,
    k=critical_n,
    use_simultaneous=True,
    time_horizon=None,
    max_time=critical_max_time,
    env=critical_env,
)
critical_build_time = time.perf_counter() - critical_start

print(f'Critical instance: n={critical_n}, c={critical_c}, seed={critical_seed}')
print(f'Build time: {critical_build_time:.4f}s')

if critical_model is None:
    print('Exact IP build failed for the selected instance.')
else:
    critical_model.setParam('OutputFlag', 1)
    critical_model.setParam('LogToConsole', 1)
    critical_model.setParam('TimeLimit', max(0.0, critical_max_time - critical_build_time))
    critical_model.optimize()

    critical_seed_nodes = [node for node in critical_graph.nodes() if critical_x[node, 0].X > 0.5]
    critical_seed_vector = np.zeros(critical_n, dtype=float)
    for node in critical_seed_nodes:
        critical_seed_vector[node] = 1.0

    critical_spread, critical_history, critical_final_x = connected_component_spread(
        critical_graph,
        critical_seed_vector,
        critical_thetas,
        max_t=1000,
    )

    print(f'Status: {critical_model.Status}')
    print(f'Objective seeds: {int(round(critical_model.objVal)) if critical_model.SolCount > 0 else None}')
    print(f'Seed nodes: {critical_seed_nodes}')
    print(f'Full spread: {critical_spread == critical_n}')
    print(f'Spread history: {critical_history}')

In [ ]:
if failed.empty:
    print('No failed instances to visualize.')
else:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    failed = failed.reset_index(drop=True)
    case_by_name = {case['name']: case for case in cases}

    def render_failed_instance(selection=0, save = 0):
        if isinstance(selection, int):
            row = failed.iloc[selection]
        else:
            row = selection

        case = case_by_name[row['name']]
        graph = case['graph']
        thetas = case['thetas']
        seed_nodes = [int(node) for node in row['seed_nodes']] if isinstance(row['seed_nodes'], list) else []

        seed_vector = np.zeros(graph.number_of_nodes(), dtype=float)
        for node in seed_nodes:
            seed_vector[node] = 1.0

        spread, spread_history, final_x = connected_component_spread(graph, seed_vector, thetas, max_t=1000)
        active_nodes = {node for node, value in enumerate(final_x) if value > 0.5}
        pos = nx.spring_layout(graph, seed=case['seed'])

        node_colors = []
        for node in graph.nodes():
            if node in seed_nodes:
                node_colors.append('#2ca02c')
            elif node in active_nodes:
                node_colors.append('#ffbb78')
            else:
                node_colors.append('#d9d9d9')

        labels = {node: f"{node}\nθ={thetas[node]}" for node in graph.nodes()}
        fig, ax = plt.subplots(figsize=(12, 8))
        nx.draw_networkx_edges(graph, pos, ax=ax, edge_color='#b0b0b0', width=1.2, alpha=0.8)
        nx.draw_networkx_nodes(
            graph,
            pos,
            ax=ax,
            node_color=node_colors,
            node_size=900,
            edgecolors='#333333',
            linewidths=1.2,
        )
        nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax, font_size=9)
        ax.set_title(
            f"{row['name']} | objective seeds={row['objective_seeds']} | spread={spread}/{graph.number_of_nodes()} | full_spread={spread == graph.number_of_nodes()}",
            pad=18,
        )
        ax.axis('off')
        ax.legend(
            handles=[
                Patch(facecolor='#2ca02c', edgecolor='#333333', label='solution seed'),
                Patch(facecolor='#ffbb78', edgecolor='#333333', label='activated by spread'),
                Patch(facecolor='#d9d9d9', edgecolor='#333333', label='inactive'),
            ],
            loc='upper left',
            bbox_to_anchor=(1.02, 1.0),
            frameon=False,
        )
        fig.text(
            0.01,
            0.01,
            f"Seed set: {seed_nodes}\nThresholds: { {node: thetas[node] for node in graph.nodes()} }\nSpread history: {spread_history}",
            ha='left',
            va='bottom',
            fontsize=9,
            family='monospace',
        )
        plt.tight_layout()
        plt.savefig(ROOT / f"failed_instance_{row['name']}.png", bbox_inches='tight') if save else None
        plt.show()
        return row

    def list_failed_instances():
        display_cols = ['name', 'n', 'c', 'seed', 'status', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread']
        return failed[display_cols]

    print('Use render_failed_instance(i) to inspect one failed case at a time.')
    print('Use list_failed_instances() to see the available indices and case names.')
    display(list_failed_instances())

    # Render all failed instances sequentially for quick review
    for i in range(len(failed)):
        print(f"\n--- Rendering failed instance {i} ---")
        render_failed_instance(i, save=1)


In [ ]:
# --- Success-case viewer ---
success = summary[summary['full_spread']].reset_index(drop=True)
if success.empty:
    print('\nNo successful instances to visualize.')
else:
    def render_success_instance(selection=0, save=0):
        if isinstance(selection, int):
            row = success.iloc[selection]
        else:
            row = selection

        case = case_by_name[row['name']]
        graph = case['graph']
        thetas = case['thetas']
        seed_nodes = [int(node) for node in row['seed_nodes']] if isinstance(row['seed_nodes'], list) else []

        seed_vector = np.zeros(graph.number_of_nodes(), dtype=float)
        for node in seed_nodes:
            seed_vector[node] = 1.0

        spread, spread_history, final_x = connected_component_spread(graph, seed_vector, thetas, max_t=1000)
        active_nodes = {node for node, value in enumerate(final_x) if value > 0.5}
        pos = nx.spring_layout(graph, seed=case['seed'])

        node_colors = []
        for node in graph.nodes():
            if node in seed_nodes:
                node_colors.append('#2ca02c')
            elif node in active_nodes:
                node_colors.append('#9ecae1')
            else:
                node_colors.append('#d9d9d9')

        labels = {node: f"{node}\nθ={thetas[node]}" for node in graph.nodes()}
        fig, ax = plt.subplots(figsize=(12, 8))
        nx.draw_networkx_edges(graph, pos, ax=ax, edge_color='#b0b0b0', width=1.2, alpha=0.8)
        nx.draw_networkx_nodes(
            graph,
            pos,
            ax=ax,
            node_color=node_colors,
            node_size=900,
            edgecolors='#333333',
            linewidths=1.2,
        )
        nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax, font_size=9)
        ax.set_title(
            f"{row['name']} | objective seeds={row['objective_seeds']} | spread={spread}/{graph.number_of_nodes()} | full_spread={spread == graph.number_of_nodes()}",
            pad=18,
        )
        ax.axis('off')
        ax.legend(
            handles=[
                Patch(facecolor='#2ca02c', edgecolor='#333333', label='solution seed'),
                Patch(facecolor='#9ecae1', edgecolor='#333333', label='activated by spread'),
                Patch(facecolor='#d9d9d9', edgecolor='#333333', label='inactive'),
            ],
            loc='upper left',
            bbox_to_anchor=(1.02, 1.0),
            frameon=False,
        )
        fig.text(
            0.01,
            0.01,
            f"Seed set: {seed_nodes}\nThresholds: { {node: thetas[node] for node in graph.nodes()} }\nSpread history: {spread_history}",
            ha='left',
            va='bottom',
            fontsize=9,
            family='monospace',
        )
        plt.tight_layout()
        plt.savefig(ROOT / f"successful_instance_{row['name']}.png", bbox_inches='tight') if save else None
        plt.show()
        return row

    def list_success_instances():
        display_cols = ['name', 'n', 'c', 'seed', 'status', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread']
        return success[display_cols]

    print('\nUse render_success_instance(i) to inspect one successful case at a time.')
    print('Use list_success_instances() to see the available indices and case names.')
    display(list_success_instances())
    # Render all successful instances sequentially for quick review
    for i in range(len(success)):
        print(f"\n--- Rendering successful instance {i} ---")
        render_success_instance(i, save=1)